# transports in JupyterLite

Both ends of this notebook are WebAssembly: the kernel is Python compiled to wasm (Pyodide),
and the widget frontend mirrors its models over the anywidget message channel — no server, no
sockets. Run the cells in order.

In [ ]:
%pip install -q transports anywidget

In [ ]:
from pydantic import BaseModel

import transports


class Device(BaseModel):
    name: str = "lamp"
    brightness: int = 60


device = Device()
session = transports.Session()
mid = session.host(device)
server = transports.Server(session)
w = transports.widget(server)  # display it: a live view of every hosted model
w

In [ ]:
# mutate the model and push the (incremental) patch to the widget above
device.brightness = 90
device.name = "beacon"
transports.sync(server)

The widget frontend can also propose edits back — server-authoritative, no wasm fetch — from
browser JavaScript via `el.transports.edit(id, ["brightness"], 75)`; an invalid value (try a
string) is rejected by pydantic in the kernel and surfaces inline via the `reject` frame.